# Lab 05 — Delta UniForm: one writer, two clocks

> One Delta table. Two commits. A Hudi reader that can visit both.

UniForm is open source and can be tested outside Databricks. Here, Delta remains the only writer and generates Hudi-compatible metadata beside its table; Hudi reads that same storage path. We compare Delta versions with Hudi instants and verify the historical row counts. This pinned OSS Hudi route is a preview: useful evidence, not a production compatibility promise.

No Docker, catalog, or remote service is required for this Hudi experiment. UniForm's Iceberg route has additional catalog/protocol prerequisites (commonly Hive Metastore); it is not silently inferred from the Hudi result. See the final limitations.

## 1 — Start a local OSS Spark session

In [ ]:
import os, re, tempfile, time, hashlib, shutil
from pathlib import Path
from delta import configure_spark_with_delta_pip
from pyspark.sql import SparkSession

DELTA_VERSION='3.3.2'
packages=[f'io.delta:delta-hudi_2.12:{DELTA_VERSION}', 'org.apache.hudi:hudi-spark3.5-bundle_2.12:0.15.0']
os.environ.setdefault('SPARK_LOCAL_IP','127.0.0.1')
b=(SparkSession.builder.master('local[2]').appName('Morlocks-UniForm-Hudi').config('spark.ui.enabled','false').config('spark.default.parallelism','2').config('spark.sql.shuffle.partitions','2').config('spark.jars.ivySettings',Path('ivysettings.xml').resolve().as_uri()).config('spark.jars.excludes','org.xerial.snappy:snappy-java,com.github.luben:zstd-jni').config('spark.serializer','org.apache.spark.serializer.KryoSerializer').config('spark.kryo.registrator','org.apache.spark.HoodieSparkKryoRegistrar').config('spark.sql.extensions','io.delta.sql.DeltaSparkSessionExtension,org.apache.spark.sql.hudi.HoodieSparkSessionExtension').config('spark.sql.catalog.spark_catalog','org.apache.spark.sql.delta.catalog.DeltaCatalog'))
spark=configure_spark_with_delta_pip(b,extra_packages=packages).getOrCreate(); spark.sparkContext.setLogLevel('ERROR')
ROOT=Path(tempfile.mkdtemp(prefix='morlocks-uniform-')).resolve(); TABLE=(ROOT/'events').as_uri()
print('Spark',spark.version,'temporary table',TABLE)

## 2 — Write twice through Delta

The first commit creates the table and enables the generated Hudi view. The second commit appends another row. Delta versions and Hudi instants are independent identifiers; do not compare their numeric shape and assume they mean the same thing.

In [ ]:
from delta.tables import DeltaTable

def event(rows):
 return spark.createDataFrame(rows,'id LONG, event_time STRING, payload STRING').selectExpr('id','to_timestamp(event_time) AS event_time','payload')

def delta_version():
 return int(DeltaTable.forPath(spark,TABLE).history().selectExpr('max(version) AS v').first()['v'])

first=event([(1,'2026-10-23 12:00:01','first commit')])
(first.write.format('delta').option('delta.columnMapping.mode','name').option('delta.universalFormat.enabledFormats','hudi').mode('overwrite').save(TABLE))
v0=delta_version()
second=event([(2,'2026-10-23 12:00:02','second commit')])
second.write.format('delta').mode('append').save(TABLE)
v1=delta_version()
print('Delta versions:',v0,'→',v1)

## 3 — Let the generated Hudi view catch up

A successful Delta write is not a promise that the secondary format is ready at the exact same millisecond. Poll the generated Hudi reader and the completed Hudi timeline instead of sleeping once and hoping.

In [ ]:
def completed_hudi_instants():
 timeline=ROOT/'events'/'.hoodie'
 found=set()
 if timeline.is_dir():
  for f in timeline.iterdir():
   m=re.match(r'^(\d{14,17})',f.name)
   completed='__COMPLETED__' in f.name and any(a in f.name for a in ('__commit__','__deltacommit__','__replacecommit__'))
   legacy=f.suffix in {'.commit','.deltacommit','.replacecommit'}
   if f.is_file() and m and (completed or legacy) and 'inflight' not in f.name.lower(): found.add(m.group(1))
 return sorted(found)

def wait_hudi(expected,timeout=90):
 deadline=time.monotonic()+timeout; last='not visible yet'
 while time.monotonic()<deadline:
  try:
   frame=spark.read.format('hudi').option('hoodie.metadata.enable','true').load(TABLE)
   instants=completed_hudi_instants()
   if frame.count()==expected and len(instants)>=expected: return frame,instants
   last=f'{frame.count()} rows, {len(instants)} completed instants'
  except Exception as e: last=f'{type(e).__name__}: {str(e)[:160]}'
  time.sleep(2)
 raise TimeoutError(f'Hudi generated view did not catch up: {last}')

hudi_latest,instants=wait_hudi(2)
print('Hudi completed instants:',instants)

## 4 — Travel back in both coordinate systems

Read Delta at its first version and Hudi at its first completed instant. Then compare each historical view with the latest state. Same logical data, different native history coordinates.

In [ ]:
delta_latest=spark.read.format('delta').load(TABLE)
results={'Delta v0':fingerprint(delta_past),'Hudi first instant':fingerprint(hudi_past),'Delta latest':fingerprint(delta_latest),'Hudi latest':fingerprint(hudi_latest)}
for label,result in results.items(): print(label, result)
print('Delta historical rows:',delta_past.select('id','payload').orderBy('id').collect())
print('Hudi historical rows:',hudi_past.select('id','payload').orderBy('id').collect())
print('Delta latest rows:',delta_latest.select('id','payload').orderBy('id').collect())
print('Hudi latest rows:',hudi_latest.select('id','payload').orderBy('id').collect())
assert results['Delta v0'][0]==results['Hudi first instant'][0]==1
assert results['Delta latest'][0]==results['Hudi latest'][0]==2
if results['Delta v0'][1]==results['Hudi first instant'][1] and results['Delta latest'][1]==results['Hudi latest'][1]:
 print('VALUE CHECK: historical and latest canonical values match across formats.')
else:
 print('VALUE CHECK FAILED: row counts match, but canonical values differ. Treat this as an interoperability defect/limitation to investigate; do not claim equivalent reads.')

## Conclusions — UniForm

**Potential strength:** one Delta writer can expose a generated Hudi reader view over the same table path; this experiment checks both historical and latest states without a second copy or second writer.

**Observed result:** row counts match in this pinned OSS preview run, but the canonical value checksums differ. The lab prints each reader's rows so the mismatch is inspectable. Treat this as a failed equivalence check, not proof of interoperable correctness; don't adopt this exact configuration until the data difference is explained and a matching digest is obtained.

**Trade-offs and limits:** conversion can be asynchronous, so poll/monitor readiness. The Hudi path tested here is an OSS preview on the pinned Delta 3.3.2 / Hudi 0.15.0 stack. Generated views are read-only and are not independent backups or cross-format atomic transactions. Delta versions and Hudi instants are distinct coordinates.

**Iceberg caveat:** Delta UniForm's Iceberg route has additional Delta protocol/column-mapping and catalog requirements. This no-service notebook intentionally does not claim Iceberg UniForm success; test that route only with its documented Hive Metastore/catalog setup. Confirm exact release support, retention, and feature compatibility before production. Keep one writer and independent backups.

In [ ]:
spark.stop(); shutil.rmtree(ROOT,ignore_errors=True)
print('Temporary UniForm table removed.')